# Rutas de vehículos: VRP

## Tipo de problema

Diseñar las rutas óptimas para una flota de camiones de reparto considerando ventanas horarias de entrega (VRPTW), capacidad de carga y restricciones de conductores.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Entera o mixta | Formular rutas y buscar soluciones mediante branch-and-cut o branch-and-price. | Alternativa |
| Heurísticas y metaheurísticas | Construir y mejorar rutas con búsqueda local, ALNS o tabú. | Alternativa |
| Enumeración | Comparar todas las rutas candidatas de esta instancia pequeña. | **Principal** |

**Enfoque utilizado:** Enumeración.

**Técnica concreta:** Comparar permutaciones y cortes en dos rutas, comprobando carga, ventanas y regreso.

[Rutas y limitaciones de la búsqueda exhaustiva: OR-Tools](https://developers.google.com/optimization/routing).

## Ejercicio propuesto

Repartir a cinco clientes con dos vehículos de capacidad 4 unidades.  
Coordenadas: depósito(0,0), C1(1,1), C2(2,0), C3(3,1), C4(0,3), C5(2,3).  
Demandas: 2,2,1,2,1. Ventanas de INICIO de servicio: [0,12] para C1-C3  
 y [0,15] para C4-C5. Servicio: 1 minuto. Salida: minuto 0; regreso <=24.  
Distancia Manhattan; velocidad 1 unidad/minuto. Se permite esperar.  
Objetivo: mínima distancia total, atendiendo una vez a cada cliente.  
Método exacto: permutar clientes y cortar la secuencia en dos rutas no vacías.  
Solo 480 candidatos; esta enumeración no es adecuada para grandes flotas.

## Método elegido para este ejercicio

**Enfoque utilizado:** Enumeración.

Enumeración exacta de las permutaciones de clientes y sus cortes en dos rutas. Se rechazan rutas que incumplen carga, ventanas de inicio de servicio o regreso al depósito. El ejemplo no añade reglas laborales de conductores.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from itertools import permutations


def resolver():
    coordenadas = ((0, 0), (1, 1), (2, 0), (3, 1), (0, 3), (2, 3))
    demanda = (0, 2, 2, 1, 2, 1)
    cierre = (24, 12, 12, 12, 15, 15)

    def distancia(a, b):
        return sum(abs(x - y) for x, y in zip(coordenadas[a], coordenadas[b]))

    def evaluar(ruta):
        if sum(demanda[c] for c in ruta) > 4:
            return None
        tiempo, recorrido, anterior, agenda = 0, 0, 0, []
        for cliente in ruta:
            tramo = distancia(anterior, cliente)
            tiempo = max(0, tiempo + tramo)
            recorrido += tramo
            if tiempo > cierre[cliente]:
                return None
            agenda.append((cliente, tiempo))
            tiempo += 1
            anterior = cliente
        regreso = distancia(anterior, 0)
        if tiempo + regreso > 24:
            return None
        return recorrido + regreso, agenda, tiempo + regreso

    mejor = float('inf'), None
    for orden in permutations(range(1, 6)):
        for corte in range(1, 5):
            rutas = (orden[:corte], orden[corte:])
            resultados = tuple(evaluar(r) for r in rutas)
            if any(r is None for r in resultados):
                continue
            total = sum(r[0] for r in resultados)
            if total < mejor[0]:
                mejor = total, resultados
    assert mejor[1] is not None
    assert sorted(c for _, agenda, _ in mejor[1] for c, _ in agenda) == list(range(1, 6))
    return mejor

## Resolución y resultados

In [2]:
total, rutas = resolver()
for vehiculo, (distancia, agenda, regreso) in enumerate(rutas, 1):
    recorrido = ' -> '.join(['Depósito'] + [f'C{c}' for c, _ in agenda] + ['Depósito'])
    print(f'Vehículo {vehiculo}: {recorrido}; distancia={distancia}')
    print(f'  Inicios de servicio: {agenda}; regreso: minuto {regreso}')
print(f'Distancia total mínima: {total}')

Vehículo 1: Depósito -> C1 -> C2 -> Depósito; distancia=6
  Inicios de servicio: [(1, 2), (2, 5)]; regreso: minuto 8
Vehículo 2: Depósito -> C3 -> C5 -> C4 -> Depósito; distancia=12
  Inicios de servicio: [(3, 4), (5, 8), (4, 11)]; regreso: minuto 15
Distancia total mínima: 18


## Interpretación

Una solución óptima usa las rutas **depósito–C1–C2–depósito** y **depósito–C3–C5–C4–depósito**. La distancia mínima total es **18**. Cada vehículo transporta 4 unidades y regresa dentro del horizonte.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.